In [1]:
import pickle

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import SparkSession

from modeling.pipelines.raw.nodes import fetch_calls_weekly, fetch_events_weekly, fetch_weather_weekly
from modeling.pipelines.target.nodes import build_target
from modeling.pipelines.features.nodes import featurize_events, featurize_lags, featurize_weather, join_features
from modeling.pipelines.tweet.nodes import compute_predict_window, build_next_week_features, compute_top_k, format_tweet
from modeling.pipelines.modeling.nodes import inference

/home/zaccosenza/code/project-311/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Hardcoded here (rather than read from conf/) so the whole run is self-contained and
# easy to tweak in-notebook while debugging — should match conf/base/parameters.yml.
CALLS_URL = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
EVENTS_URL = "https://data.cityofnewyork.us/resource/bkfu-528j.json"
EVENT_INCLUDE_TYPES = [
    "Parade", "Street Festival", "Single Block Festival", "Block Party",
    "Farmers Market", "Street Event", "Religious Event", "Plaza Event",
    "Plaza Partner Event", "Athletic Race / Tour", "Open Street Partner Event",
    "Health Fair", "Sidewalk Sale",
]
WEATHER_LAT, WEATHER_LON = 40.7812, -73.9665
WEATHER_DAILY_VARS = "temperature_2m_max,temperature_2m_min,rain_sum,snowfall_sum"
WEATHER_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_FORECAST_URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"

TARGET_COL = "tgt_calls"
FEATURE_COLS = [
    "ft_week_of_year", "ft_lag_1", "ft_lag_2", "ft_event_count",
    "ft_lag1_temp_max", "ft_lag1_temp_min", "ft_lag1_had_rain", "ft_lag1_had_snow",
    "ft_pred_temp_max", "ft_pred_temp_min", "ft_pred_had_rain", "ft_pred_had_snow",
    "ft_board_key",
]
CATEGORICAL_FEATURES = ["ft_board_key"]
LOOKBACK_WEEKS = 3
TOP_K = 5

RAW_DIR = "../data/dev/00_raw"
MODEL_PATH = "../data/dev/02_reporting/model.pkl"
MODELING_DATA_PATH = "../data/dev/02_reporting/modeling_data.parquet"

In [3]:
# Same date-window logic the real tweet pipeline uses — calls can only ever be fetched
# through today, events/weather-forecast reach further ahead to cover next week.
start_date, end_date, forecast_end_date = compute_predict_window(LOOKBACK_WEEKS)
print(f"start_date={start_date}  end_date={end_date}  forecast_end_date={forecast_end_date}")

calls = fetch_calls_weekly(start_date, end_date, CALLS_URL, RAW_DIR)
events = fetch_events_weekly(start_date, forecast_end_date, EVENTS_URL, EVENT_INCLUDE_TYPES, RAW_DIR)
weather_lag1, weather_pred = fetch_weather_weekly(
    start_date, end_date, WEATHER_LAT, WEATHER_LON, WEATHER_DAILY_VARS,
    WEATHER_ARCHIVE_URL, WEATHER_FORECAST_URL, RAW_DIR, forecast_end_date=forecast_end_date,
)
print(f"calls: {len(calls):,} rows  |  events: {len(events):,} rows")
print(f"weather_lag1 max week: {weather_lag1['week_start'].max()}  |  weather_pred max week: {weather_pred['week_start'].max()}")

start_date=2026-07-20  end_date=2026-08-10  forecast_end_date=2026-08-23
calls: 230 rows  |  events: 514 rows
weather_lag1 max week: 2026-08-17  |  weather_pred max week: 2026-08-17


In [4]:
spark = SparkSession.builder.appName("inference-notebook").master("local[*]").getOrCreate()

target = build_target(calls, TARGET_COL)
lag_features = featurize_lags(target, TARGET_COL)
event_features = featurize_events(events)
weather_features = featurize_weather(weather_lag1, weather_pred)
features = join_features(target, lag_features, event_features, weather_features, FEATURE_COLS, CATEGORICAL_FEATURES)

print(f"target: {len(target):,} rows  |  features (after dropna): {len(features):,} rows")
print(f"most recent complete week in features: {features['week_start'].max()}")
features.tail()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/10 21:22:10 WARN Utils: Your hostname, ZacPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/10 21:22:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/10 21:22:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/10 21:22:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


target: 231 rows  |  features (after dropna): 77 rows
most recent complete week in features: 2026-08-03


,board_key,week_start,tgt_calls,ft_week_of_year,ft_lag_1,ft_lag_2,ft_event_count,ft_lag1_temp_max,ft_lag1_temp_min,ft_lag1_had_rain,ft_lag1_had_snow,ft_pred_temp_max,ft_pred_temp_min,ft_pred_had_rain,ft_pred_had_snow,ft_board_key
72,Unspecified BRONX,2026-08-03,146.0,32.0,5.043425,4.919981,0.0,29.8,19.3,1.0,0.0,34.2,18.2,1.0,0.0,Unspecified BRONX
73,Unspecified BROOKLYN,2026-08-03,268.0,32.0,5.676754,5.703782,0.0,29.8,19.3,1.0,0.0,34.2,18.2,1.0,0.0,Unspecified BROOKLYN
74,Unspecified MANHATTAN,2026-08-03,270.0,32.0,5.587249,5.476464,0.0,29.8,19.3,1.0,0.0,34.2,18.2,1.0,0.0,Unspecified MANHATTAN
75,Unspecified QUEENS,2026-08-03,253.0,32.0,5.513429,5.533389,0.0,29.8,19.3,1.0,0.0,34.2,18.2,1.0,0.0,Unspecified QUEENS
76,Unspecified STATEN ISLAND,2026-08-03,24.0,32.0,3.295837,3.401197,0.0,29.8,19.3,1.0,0.0,34.2,18.2,1.0,0.0,Unspecified STATEN ISLAND


In [5]:
# modeling_data supplies the full board_key category list — must match what the model
# was trained on, not just whatever boards happen to appear in this narrow window.
modeling_data = pd.read_parquet(MODELING_DATA_PATH)
next_week_features = build_next_week_features(features, event_features, weather_features, modeling_data, TARGET_COL)

with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

print(f"next week: {next_week_features['week_start'].iloc[0]}  |  {len(next_week_features)} boards")
next_week_features[["board_key", "week_start"] + FEATURE_COLS].head()

next week: 2026-08-10  |  77 boards


,board_key,week_start,ft_week_of_year,ft_lag_1,ft_lag_2,ft_event_count,ft_lag1_temp_max,ft_lag1_temp_min,ft_lag1_had_rain,ft_lag1_had_snow,ft_pred_temp_max,ft_pred_temp_min,ft_pred_had_rain,ft_pred_had_snow,ft_board_key
0,0 Unspecified,2026-08-10,33,4.304065,4.653960,0.000000,33.3,18.4,1.0,0.0,33.1,18.5,1,0,0 Unspecified
1,01 BRONX,2026-08-10,33,6.792344,6.807935,1.386294,33.3,18.4,1.0,0.0,33.1,18.5,1,0,01 BRONX
2,01 BROOKLYN,2026-08-10,33,7.360104,7.599902,1.098612,33.3,18.4,1.0,0.0,33.1,18.5,1,0,01 BROOKLYN
3,01 MANHATTAN,2026-08-10,33,6.489205,6.584791,1.386294,33.3,18.4,1.0,0.0,33.1,18.5,1,0,01 MANHATTAN
4,01 QUEENS,2026-08-10,33,7.518064,7.542744,0.000000,33.3,18.4,1.0,0.0,33.1,18.5,1,0,01 QUEENS


In [6]:
top_k_districts = compute_top_k(model, next_week_features, FEATURE_COLS, TARGET_COL, TOP_K)
print(top_k_districts.to_string(index=False))

tweet_text = format_tweet(top_k_districts, TARGET_COL)
print("\n--- tweet preview ---")
print(tweet_text)

   board_key week_start  pred_tgt_calls
12 MANHATTAN 2026-08-10     1908.170776
   01 QUEENS 2026-08-10     1751.902100
 05 BROOKLYN 2026-08-10     1693.662354
   12 QUEENS 2026-08-10     1685.243042
 01 BROOKLYN 2026-08-10     1642.487549

--- tweet preview ---
TEST_TWEET
Week of 2026-08-10:
1. 12 MANHATTAN — 1,908 calls
2. 01 QUEENS — 1,752 calls
3. 05 BROOKLYN — 1,694 calls
4. 12 QUEENS — 1,685 calls
5. 01 BROOKLYN — 1,642 calls


In [7]:
# Full (not just top-k) prediction spread — useful for spotting an obviously-wrong
# board (e.g. a stale-fallback or missing-category issue) that wouldn't show up
# just by eyeballing the top 5.
all_scored = inference(model, next_week_features, FEATURE_COLS, TARGET_COL).sort_values(
    f"pred_{TARGET_COL}", ascending=False
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(all_scored["board_key"], all_scored[f"pred_{TARGET_COL}"], color="#93c5fd")
ax.set_xticks(range(len(all_scored)))
ax.set_xticklabels(all_scored["board_key"], rotation=90, fontsize=6)
ax.set_ylabel("predicted calls")
ax.set_title(f"Predicted calls by board — week of {next_week_features['week_start'].iloc[0]}")
plt.tight_layout()
plt.show()

/tmp/ipykernel_182296/3381569300.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
spark.stop()